In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    ExtraTreesClassifier,
    ExtraTreesRegressor
)

from Model_pipeline import TwoStagePipeline

In [4]:
def safe_spearman(y_true, y_pred):
    corr = spearmanr(y_true, y_pred).correlation
    return 0.0 if pd.isna(corr) else corr

In [2]:
mymodel = TwoStagePipeline()
mymodel.load_data()
mymodel.preprocess_dates()
mymodel.prepare_training_data()
mymodel.split_data()

Features shape: (145739, 73)
Train shape: (116591, 2)
Test shape: (29148, 1)
Training matrix shape: (116591, 72)
Positive buyers ratio: 0.3659
X_train shape: (93272, 72)
X_val shape: (23319, 72)


In [5]:
gb_clf_params = {
    "subsample": 1.0,
    "n_estimators": 700,
    "min_samples_split": 10,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "max_depth": 5,
    "learning_rate": 0.01,
    "random_state": 42
}

gb_reg_params = {
    "subsample": 0.6,
    "n_estimators": 300,
    "min_samples_split": 10,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "max_depth": 3,
    "learning_rate": 0.05,
    "random_state": 42
}

et_clf_params = {
    "n_estimators": 600,
    "min_samples_split": 10,
    "min_samples_leaf": 1,
    "max_features": 0.9,
    "max_depth": 8,
    "bootstrap": True,
    "random_state": 42,
    "n_jobs": -1,
    "class_weight": "balanced"
}

et_reg_params = {
    "n_estimators": 600,
    "min_samples_split": 10,
    "min_samples_leaf": 1,
    "max_features": 0.9,
    "max_depth": 8,
    "bootstrap": True,
    "random_state": 42,
    "n_jobs": -1
}

In [6]:
gb_clf = GradientBoostingClassifier(**gb_clf_params)
gb_reg = GradientBoostingRegressor(**gb_reg_params)

et_clf = ExtraTreesClassifier(**et_clf_params)
et_reg = ExtraTreesRegressor(**et_reg_params)

In [7]:
mask_train_buyers = mymodel.y_class_train == 1
gb_clf.fit(mymodel.X_train, mymodel.y_class_train)
gb_reg.fit(
    mymodel.X_train[mask_train_buyers],
    np.log1p(mymodel.y_reg_train[mask_train_buyers])
)

# ExtraTrees
et_clf.fit(mymodel.X_train, mymodel.y_class_train)
et_reg.fit(
    mymodel.X_train[mask_train_buyers],
    np.log1p(mymodel.y_reg_train[mask_train_buyers])
)

print("Both models trained.")

Both models trained.


In [8]:
proba_gb = gb_clf.predict_proba(mymodel.X_val)[:, 1]
reg_gb = np.expm1(gb_reg.predict(mymodel.X_val))
reg_gb = np.clip(reg_gb, 0, None)

pred_gb = np.zeros(len(proba_gb))
mask_gb = proba_gb >= 0.50   
pred_gb[mask_gb] = reg_gb[mask_gb]
pred_gb = np.clip(pred_gb, 0, None)


proba_et = et_clf.predict_proba(mymodel.X_val)[:, 1]
reg_et = np.expm1(et_reg.predict(mymodel.X_val))
reg_et = np.clip(reg_et, 0, None)

pred_et = np.zeros(len(proba_et))
mask_et = proba_et >= 0.75  
pred_et[mask_et] = reg_et[mask_et]
pred_et = np.clip(pred_et, 0, None)

print("Validation predictions ready.")
print("GB predicted buyers:", int(mask_gb.sum()))
print("ET predicted buyers:", int(mask_et.sum()))

Validation predictions ready.
GB predicted buyers: 5357
ET predicted buyers: 2718


In [9]:
mae_gb = mean_absolute_error(mymodel.y_reg_val, pred_gb)
sp_gb = safe_spearman(mymodel.y_reg_val, pred_gb)

mae_et = mean_absolute_error(mymodel.y_reg_val, pred_et)
sp_et = safe_spearman(mymodel.y_reg_val, pred_et)

print("Gradient Boosting")
print("MAE     :", round(mae_gb, 4))
print("Spearman:", round(sp_gb, 4))

print("\nExtraTrees")
print("MAE     :", round(mae_et, 4))
print("Spearman:", round(sp_et, 4))

Gradient Boosting
MAE     : 64.8797
Spearman: 0.4001

ExtraTrees
MAE     : 63.3286
Spearman: 0.3551


In [10]:
rows = []

for w_et in np.arange(0.0, 1.01, 0.1):
    w_gb = 1.0 - w_et
    pred_ensemble = w_gb * pred_gb + w_et * pred_et

    mae = mean_absolute_error(mymodel.y_reg_val, pred_ensemble)
    sp = safe_spearman(mymodel.y_reg_val, pred_ensemble)

    rows.append({
        "weight_gb": round(w_gb, 2),
        "weight_et": round(w_et, 2),
        "MAE": mae,
        "Spearman": sp
    })

ensemble_results = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
ensemble_results

,weight_gb,weight_et,MAE,Spearman
0,0.4,0.6,62.580384,0.401788
1,0.3,0.7,62.630139,0.401799
2,0.5,0.5,62.669853,0.401795
3,0.2,0.8,62.800604,0.401809
4,0.6,0.4,62.891065,0.401825
5,0.1,0.9,63.053969,0.401814
6,0.7,0.3,63.225043,0.401833
7,0.0,1.0,63.328603,0.355056
8,0.8,0.2,63.668804,0.401674
9,0.9,0.1,64.224745,0.401160


In [11]:
best_row = ensemble_results.iloc[0]

best_weight_gb = best_row["weight_gb"]
best_weight_et = best_row["weight_et"]

best_pred_ensemble = best_weight_gb * pred_gb + best_weight_et * pred_et

print("Best coarse ensemble:")
print("Weight GB:", best_weight_gb)
print("Weight ET:", best_weight_et)
print("MAE      :", round(best_row["MAE"], 4))
print("Spearman :", round(best_row["Spearman"], 4))

Best coarse ensemble:
Weight GB: 0.4
Weight ET: 0.6
MAE      : 62.5804
Spearman : 0.4018


In [12]:
start_et = max(0, best_weight_et - 0.10)
end_et = min(1, best_weight_et + 0.10)

fine_rows = []

for w_et in np.arange(start_et, end_et + 0.001, 0.02):
    w_gb = 1.0 - w_et
    pred_ensemble = w_gb * pred_gb + w_et * pred_et

    mae = mean_absolute_error(mymodel.y_reg_val, pred_ensemble)
    sp = safe_spearman(mymodel.y_reg_val, pred_ensemble)

    fine_rows.append({
        "weight_gb": round(w_gb, 2),
        "weight_et": round(w_et, 2),
        "MAE": mae,
        "Spearman": sp
    })

fine_ensemble_results = pd.DataFrame(fine_rows).sort_values("MAE").reset_index(drop=True)
fine_ensemble_results

,weight_gb,weight_et,MAE,Spearman
0,0.38,0.62,62.579090,0.401791
1,0.40,0.60,62.580384,0.401788
2,0.36,0.64,62.584039,0.401792
3,0.42,0.58,62.587488,0.401786
4,0.34,0.66,62.594707,0.401794
5,0.44,0.56,62.599337,0.401785
6,0.32,0.68,62.609581,0.401796
7,0.46,0.54,62.617676,0.401787
8,0.30,0.70,62.630139,0.401799
9,0.48,0.52,62.641100,0.401791


In [13]:
final_best_row = fine_ensemble_results.iloc[0]

final_weight_gb = final_best_row["weight_gb"]
final_weight_et = final_best_row["weight_et"]

print("Final best ensemble")
print("Weight GB:", final_weight_gb)
print("Weight ET:", final_weight_et)
print("Validation MAE     :", round(final_best_row["MAE"], 4))
print("Validation Spearman:", round(final_best_row["Spearman"], 4))

Final best ensemble
Weight GB: 0.38
Weight ET: 0.62
Validation MAE     : 62.5791
Validation Spearman: 0.4018


In [14]:
gb_clf_full = GradientBoostingClassifier(**gb_clf_params)
gb_reg_full = GradientBoostingRegressor(**gb_reg_params)

et_clf_full = ExtraTreesClassifier(**et_clf_params)
et_reg_full = ExtraTreesRegressor(**et_reg_params)

mask_full_buyers = mymodel.y_class == 1

gb_clf_full.fit(mymodel.X, mymodel.y_class)
gb_reg_full.fit(
    mymodel.X[mask_full_buyers],
    np.log1p(mymodel.y_reg[mask_full_buyers])
)

et_clf_full.fit(mymodel.X, mymodel.y_class)
et_reg_full.fit(
    mymodel.X[mask_full_buyers],
    np.log1p(mymodel.y_reg[mask_full_buyers])
)

print("Both full-data models trained.")

Both full-data models trained.


In [15]:
X_test = mymodel.prepare_test_data()
print("X_test shape:", X_test.shape)

Completely empty test feature rows: 0
X_test shape: (29148, 72)


In [16]:
proba_gb_test = gb_clf_full.predict_proba(X_test)[:, 1]
reg_gb_test = np.expm1(gb_reg_full.predict(X_test))
reg_gb_test = np.clip(reg_gb_test, 0, None)

pred_gb_test = np.zeros(len(proba_gb_test))
mask_gb_test = proba_gb_test >= 0.50
pred_gb_test[mask_gb_test] = reg_gb_test[mask_gb_test]
pred_gb_test = np.clip(pred_gb_test, 0, None)

proba_et_test = et_clf_full.predict_proba(X_test)[:, 1]
reg_et_test = np.expm1(et_reg_full.predict(X_test))
reg_et_test = np.clip(reg_et_test, 0, None)

pred_et_test = np.zeros(len(proba_et_test))
mask_et_test = proba_et_test >= 0.75
pred_et_test[mask_et_test] = reg_et_test[mask_et_test]
pred_et_test = np.clip(pred_et_test, 0, None)

print("Test predictions for both models are ready.")
print("GB predicted buyers on test:", int(mask_gb_test.sum()))
print("ET predicted buyers on test:", int(mask_et_test.sum()))

Test predictions for both models are ready.
GB predicted buyers on test: 6855
ET predicted buyers on test: 3436


In [17]:
# Ensemble on test set with best validation weights

final_pred_test = final_weight_gb * pred_gb_test + final_weight_et * pred_et_test
final_pred_test = np.clip(final_pred_test, 0, None)

submission = pd.DataFrame({
    "cust_id": mymodel.X_test_full["cust_id"],
    "predicted_revenue_2018_2019": final_pred_test
})

submission_name = "submission_ensemble_gb_et.csv"
submission.to_csv(submission_name, index=False)

print("Saved:", submission_name)
print("Prediction summary")
print("min :", final_pred_test.min())
print("max :", final_pred_test.max())
print("mean:", final_pred_test.mean())

submission.head()

Saved: submission_ensemble_gb_et.csv
Prediction summary
min : 0.0
max : 513.947629797442
mean: 30.532752993422925


,cust_id,predicted_revenue_2018_2019
0,2dfoualegmpt6x2h,339.623292
1,d2q2stjpnzld7a4r,67.961139
2,cojscuqlpylhclv2,0.000000
3,vntezlhi2ryvxk6m,175.566213
4,jgy4ytjkdr2b75wf,170.596973


In [18]:
# Clean summary table for your report

summary = pd.DataFrame([{
    "gb_threshold": 0.50,
    "et_threshold": 0.75,
    "final_weight_gb": final_weight_gb,
    "final_weight_et": final_weight_et,
    "val_mae_gb": mae_gb,
    "val_mae_et": mae_et,
    "val_mae_ensemble": final_best_row["MAE"],
    "val_spearman_gb": sp_gb,
    "val_spearman_et": sp_et,
    "val_spearman_ensemble": final_best_row["Spearman"]
}])

summary

,gb_threshold,et_threshold,final_weight_gb,final_weight_et,val_mae_gb,val_mae_et,val_mae_ensemble,val_spearman_gb,val_spearman_et,val_spearman_ensemble
0,0.5,0.75,0.38,0.62,64.879702,63.328603,62.57909,0.400058,0.355056,0.401791
